In [1]:
import pandas as pd

df_product_mix = pd.read_csv(
    "../data/sp_product_mix_winter_comparison.csv"
)

print(df_product_mix.shape)
print(df_product_mix.columns)
print(df_product_mix.head())

print("\n欠損値")
print(df_product_mix.isnull().sum())

print("\n重複行")
print(df_product_mix.duplicated().sum())

(2456, 6)
Index(['season_label', 'product_category', 'product_id', 'product_order_count',
       'item_line_count', 'product_gmv'],
      dtype='str')
  season_label product_category                        product_id  \
0  2017 Winter   bed_bath_table  f1c7f353075ce59d8a6f3cf58f419c9c   
1  2017 Winter   bed_bath_table  99a4788cb24856965c36a24e339b6058   
2  2017 Winter   bed_bath_table  c6ea63369936552872ae890c82175b4b   
3  2017 Winter   bed_bath_table  952670bc019a73f0933157ec82ff6e4f   
4  2017 Winter   bed_bath_table  2a2d22ae30e026f1893083c8405ca522   

   product_order_count  item_line_count  product_gmv  
0                   46               47      9540.00  
1                   65               68      6113.20  
2                   13               13      2734.98  
3                   11               11      2254.99  
4                   12               12      1667.00  

欠損値
season_label           0
product_category       0
product_id             0
product_order_count    0

In [2]:
df_health = df_product_mix[
    df_product_mix["product_category"] == "health_beauty"
].copy()

print(df_health.shape)

print("\n季節")
print(df_health["season_label"].value_counts())

print("\n商品数")
print(
    df_health.groupby("season_label")["product_id"]
    .nunique()
)

(708, 6)

季節
season_label
2018 Winter    535
2017 Winter    173
Name: count, dtype: int64

商品数
season_label
2017 Winter    173
2018 Winter    535
Name: product_id, dtype: int64


In [3]:
products_2017 = set(
    df_health.loc[
        df_health["season_label"] == "2017 Winter",
        "product_id"
    ]
)

products_2018 = set(
    df_health.loc[
        df_health["season_label"] == "2018 Winter",
        "product_id"
    ]
)

common_products = products_2017 & products_2018
new_products_2018 = products_2018 - products_2017
disappeared_products = products_2017 - products_2018

print("2017 Winterの商品数:", len(products_2017))
print("2018 Winterの商品数:", len(products_2018))
print("両方の冬に登場:", len(common_products))
print("2018 Winterのみ:", len(new_products_2018))
print("2017 Winterのみ:", len(disappeared_products))

2017 Winterの商品数: 173
2018 Winterの商品数: 535
両方の冬に登場: 20
2018 Winterのみ: 515
2017 Winterのみ: 153


In [4]:
health_2018_top10 = (
    df_health[
        df_health["season_label"] == "2018 Winter"
    ]
    .sort_values(
        "product_gmv",
        ascending=False
    )
    .head(10)
)

print(
    health_2018_top10[
        [
            "product_id",
            "product_order_count",
            "item_line_count",
            "product_gmv"
        ]
    ]
)

                            product_id  product_order_count  item_line_count  \
1696  bb50f2e236e5eea0100680137654686c                   12               16   
1697  19c91ef95d509ea33eda93495c4d3481                   31               31   
1698  af0a99476d96dcc1a1baa7c0d9ff6b9d                    7                7   
1699  25292482a61cb3298df8dbe15ea69daf                    2                2   
1700  f7740e5b4337a3cc4a8a204553101a85                    4                4   
1701  3fbc0ef745950c7932d5f2a446189725                   27               30   
1702  f96ff9b997681965689f78bef092fdd9                    2                2   
1703  c589625c8ccc7bfd2e8f1fb041e24c4a                   11               12   
1704  89f59eee27a6a49b9f0d887136c71cd4                    1                1   
1705  6ef77c360c5206f18e444f0d170a3a01                    1                1   

      product_gmv  
1696      5280.00  
1697      3812.69  
1698      3711.30  
1699      2999.30  
1700      2160.60  

In [5]:
health_2018_top10 = health_2018_top10.copy()

health_2018_top10["appeared_in_2017_winter"] = (
    health_2018_top10["product_id"].isin(products_2017)
)

print(
    health_2018_top10[
        [
            "product_id",
            "product_order_count",
            "item_line_count",
            "product_gmv",
            "appeared_in_2017_winter"
        ]
    ]
)

                            product_id  product_order_count  item_line_count  \
1696  bb50f2e236e5eea0100680137654686c                   12               16   
1697  19c91ef95d509ea33eda93495c4d3481                   31               31   
1698  af0a99476d96dcc1a1baa7c0d9ff6b9d                    7                7   
1699  25292482a61cb3298df8dbe15ea69daf                    2                2   
1700  f7740e5b4337a3cc4a8a204553101a85                    4                4   
1701  3fbc0ef745950c7932d5f2a446189725                   27               30   
1702  f96ff9b997681965689f78bef092fdd9                    2                2   
1703  c589625c8ccc7bfd2e8f1fb041e24c4a                   11               12   
1704  89f59eee27a6a49b9f0d887136c71cd4                    1                1   
1705  6ef77c360c5206f18e444f0d170a3a01                    1                1   

      product_gmv  appeared_in_2017_winter  
1696      5280.00                     True  
1697      3812.69            

2018年冬のSP州・health_beautyでは、GMV上位商品にも2017年冬には見られなかった商品IDが多かった

In [6]:
health_2018_total_gmv = df_health.loc[
    df_health["season_label"] == "2018 Winter",
    "product_gmv"
].sum()

top10_gmv = health_2018_top10["product_gmv"].sum()

top10_share = top10_gmv / health_2018_total_gmv

print("2018 Winter health_beauty GMV:", health_2018_total_gmv)
print("Top10 GMV:", top10_gmv)
print("Top10 share:", top10_share)

2018 Winter health_beauty GMV: 126922.30999999998
Top10 GMV: 25436.730000000003
Top10 share: 0.20041181097318514


health_beauty

商品種類
173 → 535

カテゴリ注文数
336 → 1,049

カテゴリ購入割合
7.48% → 12.17%

カテゴリ注文あたりGMV
98.14 → 120.99

カテゴリGMV
32,975.36 → 126,922.31

2017冬にも登場    20商品
2018冬のみ       515商品

2018年冬はhealth_beautyの取扱商品構成が大幅に広がり、その中でカテゴリを含む注文の割合も、カテゴリ注文あたりGMVも上昇していた

Next Question.
2018年冬のGMVのうち、2017年から継続している20商品と、2018年冬にのみ登場した515商品のどちらがどれくらいGMVを作っているか

In [7]:
health_2018 = df_health[
    df_health["season_label"] == "2018 Winter"
].copy()

health_2018["product_group"] = health_2018["product_id"].apply(
    lambda x: "Common with 2017"
    if x in products_2017
    else "2018 only"
)

product_group_summary = (
    health_2018
    .groupby("product_group")
    .agg(
        product_count=("product_id", "nunique"),
        order_count=("product_order_count", "sum"),
        item_line_count=("item_line_count", "sum"),
        gmv=("product_gmv", "sum")
    )
)

product_group_summary["gmv_share"] = (
    product_group_summary["gmv"]
    / product_group_summary["gmv"].sum()
)

print(product_group_summary)

                  product_count  order_count  item_line_count        gmv  \
product_group                                                              
2018 only                   515         1000             1089  114873.35   
Common with 2017             20           86               94   12048.96   

                  gmv_share  
product_group                
2018 only          0.905068  
Common with 2017   0.094932  


2018年冬の health_beauty GMVの大部分は、2017年冬には見られなかった商品IDから生まれている

2017年冬 → 2018年冬で、
- 商品数 173 → 535
- カテゴリ注文数 336 → 1,049
- カテゴリ購入割合 7.48% → 12.17%
- カテゴリ注文あたりGMV 98.14 → 120.99
- AOV寄与額 7.34 → 14.73
と増加。

つまり、2018年冬の health_beauty の伸びは、既存の少数商品だけが伸びたというより、商品構成の大幅な入れ替わり・拡大と強く結びついている